# 05 · Run or standby: pump and temperature control

Request exactly one **RUN** or **STANDBY** change on an identified ThermoTek T255P. RUN starts circulation and temperature regulation together. STANDBY stops both; fans may take a few seconds to stop. There is no separate pump-only control in the documented interface.

This standalone serial example demonstrates the wire exchange directly. The production `TarkChiller` API provides `set_mode()`, and the dashboard exposes both controls. It reuses the production `T255P` reply validator. It does not import sibling examples or modify the driver/dashboard.

**Prepare the filled cooling loop first.** The initial U exchange can change AUTO START to RUN even when the requested mode is STANDBY. This is not an emergency-stop mechanism. Run only one device owner: close other notebooks and the dashboard first. No hardware is contacted until the operation cell is executed and you type the requested mode.

Source: [ThermoTek T255P Rev C manual](https://neurophysics.ucsd.edu/Manuals/ThermoTek/T255P_chiller_manual.pdf), printed pp7–8 and 23–29. The mode exchange has software tests only; verify physical behavior on the equipment.

## 1. Environment and imports

Run `uv sync --locked --group notebooks` from the repository root, then launch `uv run --locked --group notebooks jupyter lab`. Choose a Python 3.12+ kernel from that environment. Run cells top to bottom. Imports open no ports.

In [1]:
import sys
from pathlib import Path

# Prefer checkout sources from the repository root or examples directory.
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src/tark_chiller/controller.py").is_file():
        sys.path.insert(0, str(candidate / "src"))
        break

In [2]:
from time import perf_counter, sleep

import serial

from tark_chiller.protocols import T255P, ProtocolError

## 2. Choose the port and requested mode

Set `PORT` to the identified serial adapter, for example `"COM9"`. Choose exactly `"RUN"` or `"STANDBY"`. The mode is confirmed before opening the port. RUN uses the equipment's existing temperature target; this example never changes it. Confirm that target and the coolant/setup limits at the panel before starting.

In [3]:
PORT = "COM9"  # Set the identified serial port, e.g. "COM9".
MODE = "RUN"  # "RUN" starts circulation and temperature control together.

## 3. Serial transactions

The operation uses RS-232 at 9600 baud, 8N1, XON/XOFF, no hardware flow control, and deasserted RTS/DTR configured while closed. Verify your adapter/cable. Each request is sent once with a three-second response budget and a 64-byte limit. Reply framing, checksum, command echo and error code must pass validation. An incomplete or invalid transaction waits out its remaining response budget before raising; no request is retried.

The documented mode frames are `.G0A5\r` for STANDBY and `.G1A6\r` for RUN. U readback uses mode **1** for STANDBY and **2** for RUN; these differ from the G qualifiers.

In [4]:
def exchange(connection, command, data=b""):
    """Send once; validate framing, checksum, command echo and device error code."""
    body = b"." + command + data
    request = body + f"{sum(body) & 0xFF:02X}\r".encode("ascii")
    connection.reset_input_buffer()
    deadline = perf_counter() + 3.0
    try:
        print(f"TX {request!r}")
        if connection.write(request) != len(request):
            raise ProtocolError("Partial serial write; outcome unknown")
        response = bytearray()
        while perf_counter() < deadline:
            response.extend(connection.read(1))
            if not response:
                continue
            if T255P.is_complete(bytes(response)):
                if perf_counter() >= deadline:
                    raise TimeoutError("Reply arrived outside the 3 s response budget")
                print(f"RX {bytes(response)!r}")
                result = T255P.reply(command, bytes(response))
                if connection.in_waiting:
                    raise ProtocolError("Unexpected trailing response bytes")
                return result
            if len(response) >= 64:
                raise ProtocolError("Response exceeded 64-byte budget")
        raise TimeoutError("No complete reply within 3 s")
    except BaseException:
        # Do not permit immediate reruns to overlap an incomplete/invalid reply.
        sleep(max(0.0, deadline - perf_counter()))
        raise

## 4. Confirm and request one mode change

After the initial U/J diagnostics, call `set_mode(MODE)`. The setter checks U itself, skips a redundant G, and verifies U readback after a real G transition. RUN accepts reported RUN or AUTO START with chiller-on=True; STANDBY requires STANDBY with chiller-on=False. The reported mode label is preserved.

Saved cell outputs are preserved from an earlier execution; restart the kernel and run from the top to load the centralized setter.


In [5]:
if not isinstance(PORT, str) or not PORT.strip():
    raise ValueError('Hardware not configured: set PORT, e.g. "COM9". No port was opened.')
if MODE not in ("RUN", "STANDBY"):
    raise ValueError('MODE must be exactly "RUN" or "STANDBY". No port was opened.')

print("Prepare the filled cooling loop. Stop other notebooks/dashboard owners first.")
print("Connection sends U and can start AUTO START, even when requesting STANDBY.")
print("RUN starts pump and temperature control; STANDBY stops both.")
if input(f"Type {MODE} to request it on {PORT}; anything else cancels: ") != MODE:
    raise ValueError("Cancelled. No port was opened.")

# Configure while closed, including RTS/DTR; do not reconfigure a live UART.
connection = serial.Serial(
    port=None,
    baudrate=9600,
    bytesize=8,
    parity="N",
    stopbits=1,
    xonxoff=True,
    rtscts=False,
    dsrdtr=False,
    timeout=0.05,
    write_timeout=3.0,
)
mode_write_started = False
try:
    connection.port = PORT
    connection.rts = False
    connection.dtr = False
    connection.open()
    protocol = T255P(allow_writes=True)

    def io(command, data):
        global mode_write_started
        if command == b"G":
            mode_write_started = True  # Track actual write attempts, including partial writes.
        return exchange(connection, command, data)

    before = protocol.get_status(io)
    alarms = protocol.get_alarms(io)
    print(f"Before: mode={before['mode']}; chiller on={before['chiller_on']}")
    print(f"Alarm summary: {before['summary_alarm']}; alarms={alarms}")
    # Alarm flags are diagnostics; the device determines whether the mode can apply.

    # The setter owns the current-state check, conditional G and readback verification.
    after = protocol.set_mode(MODE, io)
    print(f"After: mode={after['mode']}; chiller on={after['chiller_on']}")
    if not after["mode_command_sent"]:
        print(
            f"{MODE} request already satisfied; reported mode={after['mode']}, "
            f"chiller on={after['chiller_on']}. No G command sent."
        )
    else:
        print(f"Reported {MODE} matches. Check the physical pump/flow and panel separately.")
except BaseException as error:
    if mode_write_started:
        error.add_note(
            "Mode outcome may be uncertain. Inspect the panel; do not replay the request."
        )
    try:
        connection.close()
    except BaseException as cleanup:
        error.add_note(f"Serial cleanup failed: {cleanup}")
    raise
else:
    connection.close()

# 5. No retry, automatic rollback, target change or mode change during cleanup.
print("Port closed. Equipment retains its mode; disconnect is not a stop command.")

Prepare the filled cooling loop. Stop other notebooks/dashboard owners first.
Connection sends U and can start AUTO START, even when requesting STANDBY.
RUN starts pump and temperature control; STANDBY stops both.
TX b'.U83\r'
RX b'#U0001069\r'
TX b'.J78\r'
RX b'#J0000000BD\r'
Before: mode=AUTO START; chiller on=True
Alarm summary: False; alarms={'float_switch': False, 'high_temp': False, 'low_temp': False, 'sensor': False, 'eeprom': False, 'watchdog': False}
RUN request already satisfied; reported mode=AUTO START, chiller on=True. No G command sent.
Port closed. Equipment retains its mode; disconnect is not a stop command.


## 5. Check results and finish

Compare the reported mode with the panel, pump and coolant flow. In STANDBY, allow the documented few seconds for the fans to stop. RUN does not mean the liquid has reached its target.

Closing the serial port preserves the requested mode. It does not stop a running chiller, restore the preceding mode or change the temperature target. The unit can return to local control after more than ten seconds without valid traffic while retaining run/standby state; this example has no keepalive worker.

If cleanup raises, resources may remain owned: finish pending I/O, then close `connection` before rerunning. Forced kernel termination cannot guarantee cleanup. Do not automatically rerun after an uncertain mode request. Use the equipment shutdown procedure when finished.